In [ ]:
pip install streamlit python-dotenv langchain langchain-community langchain-openai pypdf faiss-cpu qdrant-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the 

Data Ingestion

In [ ]:
from google.colab import userdata
import os


os.environ["OPENAI_API_KEY"] = userdata.get("key")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"
os.environ["MODEL_NAME"] = "openai/gpt-4o-mini"
from google.colab import files
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader
import os


agreement_type = input("Enter agreement type: ")


loader = PyPDFLoader(pdf_path)
pages = loader.load()


agreement_text = "\n".join([page.page_content for page in pages])


llm = ChatOpenAI(
    model=os.environ["MODEL_NAME"],
    base_url=os.environ["OPENAI_BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"]
)


prompt = f"""
You are an expert legal document analyzer.


Analyze this document as a {agreement_type}.


Document:
{agreement_text}


Return:
1. Document summary
2. Parties involved
3. Important clauses
4. Risky clauses
5. Missing clauses
6. Obligations of each party
7. Payment or penalty terms if any
8. Confidentiality terms if any
9. Termination conditions
10. Final risk rating: Low, Medium, or High
11. Practical recommendation


Do not give legal advice. Only provide an educational document analysis.
"""


response = llm.invoke(prompt)


print(response.content)


Saving sample_service_agreement.pdf to sample_service_agreement (3).pdf
Enter agreement type: sample aggrement
### 1. Document Summary
This Service Agreement details the terms for the development and maintenance of an inventory management system between ABC Software Solutions Pvt. Ltd. and XYZ Retail Stores Pvt. Ltd. The agreement outlines the scope of services, payment terms, confidentiality, intellectual property rights, termination conditions, and dispute resolution methods.

### 2. Parties Involved
- **Party A**: ABC Software Solutions Pvt. Ltd., located in Hyderabad, India.
- **Party B**: XYZ Retail Stores Pvt. Ltd., located in Bengaluru, India.

### 3. Important Clauses
- **Purpose**: Development and maintenance of an inventory management system.
- **Services**: Includes development, testing, deployment, and 12 months of support.
- **Payment Terms**: Total cost is ₹5,00,000; structure includes an advance of ₹2,00,000 and the remaining balance upon project completion.
- **Confiden

RAG

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter


splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=120
)


chunks = splitter.split_documents(pages)


embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    base_url=os.environ["OPENAI_BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"]
)


vectorstore = FAISS.from_documents(chunks, embeddings)


retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


question = input("Ask a question from the document: ")


retrieved_docs = retriever.invoke(question)


context = "\n\n".join([doc.page_content for doc in retrieved_docs])


rag_prompt = f"""
You are a document analysis assistant.


Answer only using the given context.


Context:
{context}


Question:
{question}


Answer:
"""


rag_response = llm.invoke(rag_prompt)


print(rag_response.content)


Ask a question from the document: Who are the parties involved in the agreement
The parties involved in the agreement are ABC Software Solutions Pvt. Ltd. and XYZ Retail Stores Pvt. Ltd.


In [ ]:
!pip install -q langchain langchain-community langchain-google-genai faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 2.5 MB/s eta 0:00:00


In [16]:
!pip show langchain
!pip show langchain-community
!pip show langchain-text-splitters

Name: langchain
Version: 1.3.1
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
Name: langchain-community
Version: 0.4.2
Summary: Community contributed LangChain integrations.
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: aiohttp, httpx-sse, langchain-classic, langchain-core, langsmith, numpy, pydantic-settings, pyyaml, requests, sqlalchemy, tenacity
Required-by: 
Name: langchain-text-splitters
Version: 1.1.2
Summary: LangChain text splitting utilities
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core
Required-by: langchain-classic


In [12]:
!pip install -q langchain-text-splitters